# LU Decomposition

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/linear_systems/direct_methods/lu_decomposition.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu, solve
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## What is LU Decomposition?

Any square matrix can be decomposed into the product of a lower triangular matrix ($\mathbf{L}$) and an upper triangular matrix ($\mathbf{U}$):

$$ \mathbf{A} = \mathbf{L}\mathbf{U} $$

Where:
* $\mathbf{L}$ has $1$s on the diagonal and the multipliers used during elimination below the diagonal.
* $\mathbf{U}$ is exactly the upper-triangular matrix that remains after forward Gaussian elimination!

This means **LU Decomposition is literally just Gaussian Elimination**, but instead of throwing away the elimination steps, we save them inside the $\mathbf{L}$ matrix.

## Why store the elimination steps?

If we decompose $\mathbf{A}$ into $\mathbf{L}\mathbf{U}$, solving $\mathbf{A}\mathbf{x} = \mathbf{b}$ becomes a two-step process:

$$ (\mathbf{L}\mathbf{U})\mathbf{x} = \mathbf{b} $$
$$ \mathbf{L}(\mathbf{U}\mathbf{x}) = \mathbf{b} $$

1. Let $\mathbf{y} = \mathbf{U}\mathbf{x}$. First, solve $\mathbf{L}\mathbf{y} = \mathbf{b}$ using **forward substitution** (since $\mathbf{L}$ is lower triangular).
2. Then, solve $\mathbf{U}\mathbf{x} = \mathbf{y}$ using **backward substitution** (since $\mathbf{U}$ is upper triangular).

**The massive advantage:** If you need to solve the same system for 100 different $\mathbf{b}$ vectors (e.g. testing different loads on a bridge), you only have to perform the expensive $\mathcal{O}(n^3)$ elimination once! The forward and backward substitutions only take $\mathcal{O}(n^2)$.

## PLU Decomposition in Python

In reality, standard LU decomposition fails if a zero pivot is encountered. Robust algorithms use pivoting (row swapping), which introduces a Permutation matrix ($\mathbf{P}$):

$$ \mathbf{A} = \mathbf{P}\mathbf{L}\mathbf{U} $$

Let's decompose a matrix using Scipy!

In [ ]:
A = np.array([[ 4, -2,  1],
              [-2,  4, -2],
              [ 1, -2,  4]])

# Calculate the PLU decomposition
P, L, U = lu(A)

print("Permutation Matrix P:\n", P)
print("\nLower Triangular L:\n", L)
print("\nUpper Triangular U:\n", U)

# Reconstruct A to verify
print("\nReconstructed A (P @ L @ U):\n", P @ L @ U)

## The Danger of "Fill-in"

If solving multiple $\mathbf{b}$ vectors is the goal, why don't we just calculate the exact inverse $\mathbf{A}^{-1}$ once and just do $\mathbf{x} = \mathbf{A}^{-1}\mathbf{b}$?

The answer is **Fill-in**. 

**Fill-in** is the phenomenon where, during computation, zeros in a sparse matrix become non-zeros. This destroys the sparsity pattern, requires massive amounts of memory, and drastically slows down the computation.

Let's look at what happens when you try to take the inverse of a sparse banded matrix.

In [ ]:
# Create a sparse tridiagonal matrix
n = 10
A = np.zeros((n, n))
np.fill_diagonal(A, 2)
np.fill_diagonal(A[1:], -1)
np.fill_diagonal(A[:, 1:], -1)

# Calculate its exact inverse
A_inv = np.linalg.inv(A)

# Plot their sparsity patterns side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.spy(A)
ax1.set_title("Original Banded Matrix A (Sparse!)")
ax2.spy(A_inv)
ax2.set_title("Inverse Matrix A^-1 (Completely Dense!)")
plt.show()

### Yikes!
The exact inverse of a sparse banded matrix is almost always **completely dense**. If this matrix was $10,000 \times 10,000$, storing the inverse would instantly crash your computer's RAM.

### The LU Solution
This is why we use LU decomposition! 
* The LU decomposition of a banded matrix produces $\mathbf{L}$ and $\mathbf{U}$ matrices that are *also* banded! 
* LU decomposition completely prevents catastrophic fill-in, preserving our memory and computational speed.

## Standard Software Solvers

In practice, you rarely write your own LU decomposition solver. 

Numpy and Scipy both default to highly optimized (P)LU decomposition under the hood when you call their solve functions:
* `np.linalg.solve(A, b)`
* `scipy.linalg.solve(A, b)`

In [ ]:
A = np.array([[2, -1, 0],
              [-1, 2, -1],
              [0, -1, 2]])
b = np.array([1, 2, 3])

x_numpy = np.linalg.solve(A, b)
print("Solution using numpy.linalg.solve:\n", x_numpy)

x_scipy = solve(A, b)
print("\nSolution using scipy.linalg.solve:\n", x_scipy)

## Sparse Software Solvers

If you know your matrix is sparse, you must explicitly use a sparse solver to prevent it from being treated as dense. 

The advent of distributed computing motivated incredible algorithms suited for massive sparse systems:
* **PARADISO** (PARallel Direct SOlver)
* **SuperLU** (Supernodal LU)
* **UMFPACK** (Unsymmetric-pattern MultiFrontal method)

Scipy provides `scipy.sparse.linalg.spsolve`, which routes to SuperLU or UMFPACK automatically! Let's see the speed difference.

In [ ]:
# Generate a 200x200 sparse tridiagonal matrix
n = 200
main_diag = np.full(n, 2)
upper_diag = np.full(n - 1, -1)
lower_diag = np.full(n - 1, -1)
A_sparse = diags([lower_diag, main_diag, upper_diag], offsets=[-1, 0, 1], format='csr')

b = np.random.rand(n)

# Convert to dense format for comparison
A_dense = A_sparse.toarray()

print("Timing the Sparse Solver:")
%timeit spsolve(A_sparse, b)

print("\nTiming the Dense Solver:")
%timeit solve(A_dense, b)

## Dr. Mike's Tips!

* Direct solvers are your 'black box' for robust solution. They are the workhorse!
* Unless you have a performance problem, you should use a direct solver.
* NEVER calculate the inverse of a matrix directly unless you strictly need the elements of the inverse itself. Always use a solve function.